In [ ]:
from jetutils.plots import *
from jetutils.definitions import *
from jetutils.data import *
from jetutils.jet_finding import coarsen_da, haversine_from_dl, average_jet_categories, jet_integral_haversine, add_normals, jet_position_as_da, extract_features
from jetutils.anyspell import *
import altair as alt

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

# CESM things

In [ ]:
props_cesm = pl.read_parquet(
    "/Users/bandelol/Documents/code_local/data/props_as_df_cesm.parquet"
)
props_era5 = pl.read_parquet(
    "/Users/bandelol/Documents/code_local/data/props_as_df.parquet"
)

In [ ]:
def periodic_rolling(df: pl.DataFrame, winsize: int, data_vars: list):
    df = df.cast({"dayofyear": pl.Int32})
    halfwinsize = winsize // 2
    other_columns = get_index_columns(df, ("member", "jet"))
    descending = [False, *[col == "jet" for col in other_columns]]
    len_ = [df[col].unique().len() for col in other_columns]
    len_ = np.prod(len_)
    max_doy = df["dayofyear"].max()
    df = df.sort(["dayofyear", *other_columns], descending=descending)
    df = pl.concat(
        [
            df.tail(halfwinsize * len_).with_columns(pl.col("dayofyear") - max_doy),
            df,
            df.head(halfwinsize * len_).with_columns(pl.col("dayofyear") + max_doy),
        ]
    )
    df = df.rolling(
        pl.col("dayofyear"),
        period=f"{winsize}i",
        offset=f"-{halfwinsize + 1}i",
        group_by=other_columns,
    ).agg(*[pl.col(col).mean() for col in data_vars])
    df = df.sort(["dayofyear", *other_columns], descending=descending)
    df = df.slice(halfwinsize * len_, max_doy * len_)
    return df

In [ ]:
from matplotlib.dates import DateFormatter, MonthLocator


def plot_seasonal(
    data_vars: list,
    props_as_df: pl.DataFrame,
    nrows: int = 3,
    ncols: int = 4,
    clear: bool = True,
    suffix: str = "",
):
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 3.5, nrows * 2.4),
        tight_layout=True,
        sharex="all",
    )
    axes = axes.flatten()
    jets = props_as_df["jet"].unique().to_numpy()
    gb = props_as_df.group_by(
        [pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("jet")],
        maintain_order=True,
    )
    means = gb.mean()
    means = periodic_rolling(means, 15, data_vars)
    x = means["dayofyear"].unique()
    medians = gb.median()
    medians = periodic_rolling(medians, 15, data_vars)
    q025 = gb.quantile(0.25)
    q075 = gb.quantile(0.75)
    for varname, ax in zip(data_vars, axes.ravel()):
        dji = varname == "double_jet_index"
        ys = means[varname].to_numpy().reshape(366, 2)
        qs = np.stack(
            [
                q025[varname].to_numpy().reshape(366, 2),
                q075[varname].to_numpy().reshape(366, 2),
            ],
            axis=2,
        )
        median = medians[varname].to_numpy().reshape(366, 2)
        for i in range(2):
            color = "black" if dji else COLORS[2 - i]
            ax.fill_between(
                x, qs[:, i, 0], qs[:, i, 1], color=color, alpha=0.2, zorder=-10
            )
            ax.plot(x, median[:, i], lw=2, color=color, ls="dotted", zorder=0)
            ax.plot(x, ys[:, i], lw=3, color=color, label=jets[i], zorder=10)
            if dji:
                break
        ax.set_title(
            f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]"
        )
        ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
        ax.xaxis.set_major_formatter(DateFormatter("%b"))
        ax.set_xlim(min(x), max(x))
        if varname == "mean_lev":
            ax.invert_yaxis()
        ylim = ax.get_ylim()
        # wherex = np.isin(x.month, [6, 7, 8])
        # ax.fill_between(x, *ylim, where=wherex, alpha=0.1, color="black", zorder=-10)
        ax.set_ylim(ylim)
    axes.ravel()[0].legend().set_zorder(102)
    # plt.savefig(f"{FIGURES}/jet_props_misc/jet_props_seasonal{suffix}.png")

In [ ]:
props_era5

In [ ]:
data_vars = [
    "mean_lat",
    "mean_lev",
    "s_star",
    "tilt",
    "wavinessDC16",
    "width",
]
plot_seasonal(data_vars, props_era5, nrows=2, ncols=3, clear=False, suffix="_subset")

In [ ]:
from matplotlib.dates import DateFormatter, MonthLocator

nrows = 2
ncols = 3
winsize = 60
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
axes = axes.flatten()

n_years = props_era5["time"].dt.year().n_unique()
rng = np.random.default_rng()
n_bootstraps = 500
bootstrap_len = 4
num_blocks = n_years // bootstrap_len

sample_indices = rng.choice(
    n_years - bootstrap_len - 1, size=(n_bootstraps, n_years // bootstrap_len)
)
sample_indices = sample_indices[..., None] + np.arange(bootstrap_len)[None, None, :]
sample_indices = sample_indices.reshape(n_bootstraps, num_blocks * bootstrap_len)
sample_indices = np.append(sample_indices, np.arange(n_years)[None, :], axis=0)
sample_indices = sample_indices.flatten()

# data_vars = [col for col in props_era5.columns if col not in ["time", "jet", "year", "flag"]]
props_as_df_daily = (
    props_era5.group_by_dynamic(pl.col("time"), every="1d", group_by="jet")
    .agg(**{data_var: pl.col(data_var).mean() for data_var in data_vars})
    .sort([pl.col("time"), pl.col("jet")])
)
props_as_df_daily = props_as_df_daily.filter(pl.col("time").dt.ordinal_day() < 366)
ts_bootstrapped = (
    props_as_df_daily.group_by(
        [pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("jet")],
        maintain_order=True,
    )
    .agg(
        **{data_var: pl.col(data_var).gather(sample_indices) for data_var in data_vars},
        year=pl.col("time").dt.year().gather(sample_indices),
        inside_index=pl.int_range(len(sample_indices)) % 64,
        sample_index=pl.int_range(len(sample_indices)) // 64,
    )
    .explode([*data_vars, "year", "inside_index", "sample_index"])
)
slopes = ts_bootstrapped.group_by(
    ["dayofyear", "sample_index", "jet"], maintain_order=True
).agg(
    **{
        data_var: pl.col(data_var)
        .least_squares.ols(
            pl.col("inside_index"),
            mode="coefficients",
            add_intercept=True,
            null_policy="drop",
        )
        .struct.field("inside_index")
        for data_var in data_vars
    }
)
pvals = slopes.group_by(["dayofyear", "jet"], maintain_order=True).agg(
    **{
        data_var: pl.col(data_var)
        .head(n_bootstraps)
        .sort()
        .search_sorted(pl.col(data_var).get(-1))
        / n_bootstraps
        for data_var in data_vars
    }
)

ys = slopes.filter(pl.col("sample_index") == n_bootstraps)
ys = periodic_rolling(ys, winsize, data_vars)

for varname, ax in zip(data_vars, axes.ravel()):
    dji = varname == "double_jet_index"
    if varname == "mean_lev":
        ax.invert_yaxis()
    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    dji = varname == "double_jet_index"
    for i, jet in enumerate(["STJ", "EDJ"]):
        color = "black" if dji else COLORS[2 - i]
        y = ys.filter(pl.col("jet") == jet)[varname]
        ps = pvals.filter(pl.col("jet") == jet)[varname]
        x = np.arange(len(y))
        ax.plot(x, y, lw=3, color=color, label=jet, zorder=10)
        filter_ = (ps > 0.975) | (ps < 0.025)
        ax.scatter(
            x[filter_],
            y.filter(filter_),
            marker="x",
            color=color,
            s=100,
            linewidths=2,
            zorder=10,
        )
        if dji:
            break
    ax.set_title(f"{PRETTIER_VARNAME[varname]}, [{UNITS.get(varname, '1')}/year]")
    ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
    ax.xaxis.set_major_formatter(DateFormatter("%b"))
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()
    ylim = ax.get_ylim()
    ax.set_ylim(ylim)
    ax.grid(True)
axes.ravel()[0].legend().set_zorder(102)
# plt.savefig(f"{FIGURES}/jet_props_misc/dayofyear_trends_{winsize=}.png")

In [ ]:
data_vars = [col for col in props_cesm.columns if col not in ["member", "time", "jet"]]

In [ ]:
props_cesm = props_cesm.with_columns(future=pl.col("time").dt.year() > 2020)
monthly_seasonalities_cesm = props_cesm.group_by(
    ["future", "member", "jet", pl.col("time").dt.month().alias("month")],
    maintain_order=True,
).mean()

props_era5 = props_era5.filter(
    pl.col("time").dt.year().is_in(props_cesm["time"].dt.year().unique()),
    pl.col("time").dt.hour() == 12,
)
monthly_seasonalities_era5 = props_era5.group_by(
    ["jet", pl.col("time").dt.month().alias("month")],
    maintain_order=True,
).mean()

In [ ]:
def ttest_df(df):
    test_results = ttest_ind(
        df[::2, data_vars],
        df[1::2, data_vars],
        axis=0,
        equal_var=False,
        nan_policy="omit",
    )
    index_columns = df[:3, ["jet", "month"]]
    deltam = (df[1::2, data_vars].to_numpy() - df[::2, data_vars].to_numpy()).mean(
        axis=0
    )
    index_columns = index_columns.with_columns(
        result=np.array(["deltam", "statistic", "pvalue"])
    )
    df = pl.DataFrame(
        {
            "deltam": deltam,
            "statistic": test_results.statistic,
            "pvalue": test_results.pvalue,
        }
    ).transpose(column_names=data_vars)
    return pl.concat([index_columns, df], how="horizontal")


gb = monthly_seasonalities_cesm.group_by("jet", "month", maintain_order=True)
ttest_res = gb.map_groups(ttest_df)

In [ ]:
data_vars_plot = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_s",
    "s_star",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "persistence",
    "com_speed",
]
ncols, nrows = 4, 4
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
axes = axes.ravel()
for varname, ax in zip(data_vars_plot, axes):
    y = ttest_res.filter(pl.col("jet") == "EDJ", pl.col("result") == "deltam")[varname]
    p = ttest_res.filter(pl.col("jet") == "EDJ", pl.col("result") == "pvalue")[varname]
    x = ttest_res["month"].unique()
    signif = p < 0.01
    ax.plot(x, y, color=COLORS[1], lw=2)
    ax.scatter(
        x.filter(signif), y.filter(signif), color=COLORS[1], marker="s", s=70, lw=2
    )

    y = ttest_res.filter(pl.col("jet") == "STJ", pl.col("result") == "deltam")[varname]
    p = ttest_res.filter(pl.col("jet") == "STJ", pl.col("result") == "pvalue")[varname]
    x = ttest_res["month"].unique()
    signif = p < 0.01
    ax.plot(x, y, color=COLORS[2], lw=2)
    ax.scatter(
        x.filter(signif), y.filter(signif), color=COLORS[2], marker="s", s=70, lw=2
    )
    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    ax.set_xticks(range(3, 13, 3), labels=["Mar", "Jun", "Sep", "Dec"])
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()

In [ ]:
data_vars_plot = [
    "mean_lat",
    "mean_lev",
    "s_star",
    "tilt",
    "wavinessDC16",
    "width",
]
ncols, nrows = 3, 2
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
axes = axes.ravel()
for varname, ax in zip(data_vars_plot, axes):
    y = ttest_res.filter(pl.col("jet") == "EDJ", pl.col("result") == "deltam")[varname]
    p = ttest_res.filter(pl.col("jet") == "EDJ", pl.col("result") == "pvalue")[varname]
    x = ttest_res["month"].unique()
    signif = p < 0.01
    ax.plot(x, y, color=COLORS[1], lw=2, label="EDJ")
    ax.scatter(
        x.filter(signif), y.filter(signif), color=COLORS[1], marker="s", s=70, lw=2
    )

    y = ttest_res.filter(pl.col("jet") == "STJ", pl.col("result") == "deltam")[varname]
    p = ttest_res.filter(pl.col("jet") == "STJ", pl.col("result") == "pvalue")[varname]
    x = ttest_res["month"].unique()
    signif = p < 0.01
    ax.plot(x, y, color=COLORS[2], lw=2, label="STJ")
    ax.scatter(
        x.filter(signif), y.filter(signif), color=COLORS[2], marker="s", s=70, lw=2
    )
    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    ax.set_xticks(range(3, 13, 3), labels=["Mar", "Jun", "Sep", "Dec"])
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()
axes[-1].legend()

In [ ]:
from typing import Literal


def add_jitter(
    x: pl.DataFrame,
    intensity: float = 0.1,
    side: Literal["left", "right", "both"] = "both",
):
    x = x.to_numpy()
    rng = np.random.default_rng()
    match side:
        case "left":
            jitter = -rng.random(len(x)) * intensity
        case "right":
            jitter = rng.random(len(x)) * intensity
        case "both":
            jitter = rng.random(len(x)) * intensity * 2 - intensity
    x = x + jitter
    return x

In [ ]:
index_columns = ["future", "member", "jet", "month"]
data_vars_plot = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_s",
    "s_star",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "persistence",
    "com_speed",
]
ncols, nrows = 4, 4
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
dx = 0.1
jitter = dx
axes = axes.ravel()
for varname, ax in zip(data_vars_plot, axes):
    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "EDJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x - dx, jitter, "left"),
        y[varname],
        marker="o",
        facecolor="none",
        edgecolor=COLORS[1],
    )

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "EDJ", pl.col("future") == True
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x + dx, jitter, "right"),
        y[varname],
        marker="s",
        facecolor="none",
        edgecolor=COLORS[0],
    )

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "STJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x - dx, jitter, "left"),
        y[varname],
        marker="o",
        facecolor="none",
        edgecolor=COLORS[2],
    )

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "STJ", pl.col("future") == True
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x + dx, jitter, "right"),
        y[varname],
        marker="s",
        facecolor="none",
        edgecolor=COLORS[3],
    )
    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    ax.set_xticks(range(3, 13, 3), labels=["Mar", "Jun", "Sep", "Dec"])
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()

In [ ]:
from matplotlib.lines import Line2D

index_columns = ["future", "member", "jet", "month"]
data_vars_plot = [
    "mean_lat",
    "mean_lev",
    "s_star",
    "tilt",
    "wavinessDC16",
    "width",
]
ncols, nrows = 3, 2
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
dx = 0.1
jitter = dx
axes = axes.ravel()
for varname, ax in zip(data_vars_plot, axes):
    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "EDJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x - dx, jitter, "left"),
        y[varname],
        marker="o",
        facecolor="none",
        edgecolor=COLORS[1],
    )

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "EDJ", pl.col("future") == True
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x + dx, jitter, "right"),
        y[varname],
        marker="s",
        facecolor="none",
        edgecolor=COLORS[0],
    )

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "STJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x - dx, jitter, "left"),
        y[varname],
        marker="o",
        facecolor="none",
        edgecolor=COLORS[2],
    )

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "STJ", pl.col("future") == True
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x + dx, jitter, "right"),
        y[varname],
        marker="s",
        facecolor="none",
        edgecolor=COLORS[3],
    )
    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    ax.set_xticks(range(3, 13, 3), labels=["Mar", "Jun", "Sep", "Dec"])
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()
legend_elements = [
    Line2D(
        [0],
        [0],
        color=COLORS[2],
        lw=0,
        marker="o",
        label="STJ",
        markerfacecolor="none",
        markeredgewidth=2,
    ),
    Line2D(
        [0],
        [0],
        color=COLORS[1],
        lw=0,
        marker="o",
        label="EDJ",
        markerfacecolor="none",
        markeredgewidth=2,
    ),
    Line2D(
        [0],
        [0],
        color="black",
        lw=0,
        marker="o",
        label="Present",
        markerfacecolor="none",
        markeredgewidth=2,
    ),
    Line2D(
        [0],
        [0],
        color="black",
        lw=0,
        marker="s",
        label="Future",
        markerfacecolor="none",
        markeredgewidth=2,
    ),
]
axes[-1].legend(handles=legend_elements, ncol=1, loc="lower left")

In [ ]:
index_columns = ["future", "member", "jet", "month"]
ata_vars_plot = [
    "mean_lon",
    "mean_lat",
    "mean_lev",
    "mean_s",
    "s_star",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessR16",
    "wavinessDC16",
    "wavinessFV15",
    "width",
    "int",
    "int_over_europe",
    "persistence",
    "com_speed",
]
ncols, nrows = 4, 4
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
dx = 0.15
jitter = 0.1
axes = axes.ravel()
for varname, ax in zip(data_vars_plot, axes):
    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "EDJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x - dx, jitter, "both"),
        y[varname],
        marker=".",
        facecolor="none",
        edgecolor=COLORS[1],
    )
    y = y.group_by("month", maintain_order=True).median()
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[1], lw=2, ls="dashed")

    y = monthly_seasonalities_era5.filter(pl.col("jet") == "EDJ")
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[1], lw=2)

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "STJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x + dx, jitter, "both"),
        y[varname],
        marker=".",
        facecolor="none",
        edgecolor=COLORS[2],
    )
    y = y.group_by("month", maintain_order=True).median()
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[2], lw=2, ls="dashed")

    y = monthly_seasonalities_era5.filter(pl.col("jet") == "STJ")
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[2], lw=2)

    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    ax.set_xticks(range(3, 13, 3), labels=["Mar", "Jun", "Sep", "Dec"])
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()

In [ ]:
index_columns = ["future", "member", "jet", "month"]
data_vars_plot = [
    "mean_lat",
    "mean_lev",
    "s_star",
    "tilt",
    "wavinessDC16",
    "width",
]
ncols, nrows = 3, 2
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=True, sharex="all"
)
dx = 0.15
jitter = 0.1
axes = axes.ravel()
for varname, ax in zip(data_vars_plot, axes):
    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "EDJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x - dx, jitter, "both"),
        y[varname],
        marker=".",
        facecolor="none",
        edgecolor=COLORS[1],
    )
    y = y.group_by("month", maintain_order=True).median()
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[1], lw=2, ls="dashed")

    y = monthly_seasonalities_era5.filter(pl.col("jet") == "EDJ")
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[1], lw=2)

    y = monthly_seasonalities_cesm.filter(
        pl.col("jet") == "STJ", pl.col("future") == False
    )
    x = y["month"]
    ax.scatter(
        add_jitter(x + dx, jitter, "both"),
        y[varname],
        marker=".",
        facecolor="none",
        edgecolor=COLORS[2],
    )
    y = y.group_by("month", maintain_order=True).median()
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[2], lw=2, ls="dashed")

    y = monthly_seasonalities_era5.filter(pl.col("jet") == "STJ")
    x = y["month"]
    ax.plot(x, y[varname], color=COLORS[2], lw=2)

    ax.set_title(f"{PRETTIER_VARNAME.get(varname, varname)} [{UNITS.get(varname, '')}]")
    ax.set_xticks(range(3, 13, 3), labels=["Mar", "Jun", "Sep", "Dec"])
    ax.set_xlim(min(x), max(x))
    if varname == "mean_lev":
        ax.invert_yaxis()

legend_elements = [
    Line2D([0], [0], color=COLORS[2], lw=2, label="STJ"),
    Line2D([0], [0], color=COLORS[1], lw=2, label="EDJ"),
    Line2D([0], [0], color="black", lw=2, label="ERA5"),
    Line2D(
        [0],
        [0],
        color="black",
        lw=2,
        ls="dashed",
        marker=".",
        label="Future",
        markerfacecolor="none",
        markeredgewidth=1.5,
        markersize=9,
    ),
]
axes[-1].legend(handles=legend_elements)